# 분석 개요

1. 결제 데이터 기반 유저 코호트
2. 지점 이용기록과 연결
3. 돈을 만드는 지점 vs 이탈 시키는지점

# DB 데이터 SQL

In [ ]:

# 2. SSH Tunnel 연결 및 DB 접속
with SSHTunnelForwarder(
    (ssh_host, 22),
    ssh_username=ssh_username,
    ssh_pkey=pem_path,
    remote_bind_address=(db_host, db_port),
    local_bind_address=('localhost', 5433)
) as tunnel:

    # 3. DB 연결 (로컬 포트를 통해)
    conn = psycopg2.connect(
        host='localhost',
        port=5433,
        database=db_name,
        user=db_user,
        password=db_password
    )

    # 4. 쿼리 실행
    df_payment = pd.read_sql("""
                        SELECT

                        TO_CHAR(ph.requested_at AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS p_date,
                        TO_CHAR(ph.requested_at AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD HH:MM:SS') AS p_time,
                        c.contract_uid transaction_id,
                        c.client_uid uid,
                        c.status,
                        ph.payment_status AS ph_status,
                        c.product_name,
                        pp.name product_period,
                        TO_CHAR(c.start_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS start_date,
                        TO_CHAR(c.initial_end_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS initial_end_date,
                        TO_CHAR(c.actual_end_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS actual_end_date,
                        c.actual_price,
                        TO_CHAR(c.created_at AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD HH24:MI:SS') AS created_at,
                        c.is_migrated,
                        p.order_id,
                        u.phone_number
                        FROM contract c
                        LEFT JOIN payment p
                        ON c.contract_uid = p.contract_payment_uid
                        LEFT JOIN price_policy pp
                        ON c.price_policy_uid = pp.price_policy_uid
                        LEFT JOIN client u
                        ON c.client_uid = u.client_uid
                        LEFT JOIN payment_history ph
                        ON contract_uid = ph.payment_uid
                        where actual_price > 0
                        ORDER BY ph.requested_at ASC;

                    """, conn)
    conn.close()
df_payment['product_name'] = df_payment['product_name'] + ' ' + df_payment['product_period'].astype(str)
df_payment = df_payment.drop(columns=['product_period'])
df_payment.tail()

## 유니버스 데이터 이전 삭제

In [ ]:
import pandas as pd

# 1. start_date가 datetime 형식인지 다시 한번 확인 (안 되어있을 경우를 대비)
df_payment['start_date'] = pd.to_datetime(df_payment['start_date'])
df_payment['p_date'] = pd.to_datetime(df_payment['p_date'])

# 2. 2025-06-11부터의 데이터만 남기기 (6월 10일 포함 이전 데이터 삭제)
# 기준일: 2025-06-11
df_payment = df_payment[df_payment['start_date'] >= '2025-06-11'].reset_index(drop=True)
df_payment = df_payment[df_payment['p_date'] >= '2025-06-11'].reset_index(drop=True)
df_payment = df_payment[df_payment['p_date'] <= '2026-04-30'].reset_index(drop=True)


# 3. 결과 확인
df_payment.head()

In [ ]:
df_payment.groupby('status')['uid'].nunique().reset_index()

##코호트분류를 위한 구매형태 전처리

In [ ]:


# 1. 날짜 데이터 형식 변환
df_payment['start_date'] = pd.to_datetime(df_payment['start_date'])
df_payment['actual_end_date'] = pd.to_datetime(df_payment['actual_end_date'])

# 2. [계약별 기간] 계산 (단위: 일)
# 환불건(CANCELED)의 경우 start_date부터 환불일(actual_end_date)까지의 이용 기간이 됨
df_payment['contract_duration'] = (df_payment['actual_end_date'] - df_payment['start_date']).dt.days+1

# 3. [유저별 집계] 데이터 산출 (phone_number 기준)
user_group = df_payment.groupby('phone_number')

# (1) 총 거래액: ph_status가 'DONE'인 금액만 합산
total_rev = df_payment[df_payment['ph_status'] == 'DONE'].groupby('phone_number')['actual_price'].sum()

# (2) 총 구매횟수: ph_status가 'DONE'인 건수만 카운트
total_cnt = df_payment[df_payment['ph_status'] == 'DONE'].groupby('phone_number').size()

# (3) 총 계약기간: 유저별 모든 계약 기간(이용 기간)의 합계
total_dur = user_group['contract_duration'].sum()

# (4) 최초 구매일: 코호트 분류의 기준점
first_date = user_group['start_date'].min()

# 4. 원본 df_payment에 컬럼 붙이기 (Map 함수 활용)
df_payment['total_revenue'] = df_payment['phone_number'].map(total_rev).fillna(0)
df_payment['total_purchase_cnt'] = df_payment['phone_number'].map(total_cnt).fillna(0)
df_payment['total_contract_duration'] = df_payment['phone_number'].map(total_dur).fillna(0)
df_payment['first_purchase_date'] = df_payment['phone_number'].map(first_date)

# 5. [첫 구매 여부] 생성
# 현재 행의 시작일이 유저의 최초 구매일과 같으면 True
df_payment['is_first_purchase'] = df_payment['start_date'] == df_payment['first_purchase_date']

# 6. [환불 전 이용 기간] 별도 표시
# ph_status가 CANCELED인 행에 대해서만 의미를 가짐
df_payment['days_before_cancel'] = df_payment.apply(
    lambda x: x['contract_duration'] if x['ph_status'] == 'CANCELED' else None, axis=1
)

# 결과 확인
df_payment.head()

In [ ]:
import re

def clean_product_name(name):
    if not isinstance(name, str):
        return name

    # 1. 기본 명칭 통일 (무제한 -> 무제한 패스, 주말&야간 -> 주말&야간 패스)
    # 이미 '패스'가 포함되어 있으면 중복해서 붙이지 않음
    if '무제한' in name and '패스' not in name:
        name = name.replace('무제한', '무제한 패스')
    if '주말&야간' in name and '패스' not in name:
        name = name.replace('주말&야간', '주말&야간 패스')

    # 2. 산술 연산 처리 (6x2 -> 12, 6+6 -> 12, 12+1 -> 13)
    # 곱셈 패턴 처리 (숫자 x 숫자)
    mult_match = re.search(r'(\d+)\s*[xX]\s*(\d+)', name)
    if mult_match:
        calc_val = int(mult_match.group(1)) * int(mult_match.group(2))
        name = re.sub(r'\d+\s*[xX]\s*\d+', str(calc_val), name)

    # 덧셈 패턴 처리 (숫자 + 숫자)
    plus_match = re.search(r'(\d+)\s*\+\s*(\d+)', name)
    if plus_match:
        calc_val = int(plus_match.group(1)) + int(plus_match.group(2))
        name = re.sub(r'\d+\s*\+\s*\d+', str(calc_val), name)

    # 3. 기타 불필요한 기호 및 공백 정리
    name = name.replace('+', ' ') # 숫자가 아닌 곳에 붙은 + 제거
    name = re.sub(r'\s+', ' ', name) # 중복 공백을 하나로 합침

    return name.strip()

# 데이터프레임 적용 예시
df_payment['product_name_clean'] = df_payment['product_name'].apply(clean_product_name)
df_payment.head()

In [ ]:
# 전체 기록(df_payment)에서 유저별로 '차감형 패스' 구매 이력이 한 번이라도 있는지 확인하여 컬럼 생성
df_payment['is_count_pass_user'] = df_payment.groupby('phone_number')['product_name_clean'].transform(
    lambda x: (x.str.contains('회', na=False) & ~x.str.contains(' 1회|1회권', na=False)).any()
)

In [ ]:
df_payment.groupby('contract_duration')['phone_number'].nunique()

In [ ]:
df_payment['product_name_clean'].unique()

In [ ]:
df_payment.groupby('contract_duration')['phone_number'].nunique()

In [ ]:
df_payment.head()

In [ ]:
# 1. 날짜 데이터 형식 변환 (재확인)
df_payment['p_date'] = pd.to_datetime(df_payment['p_date'])
df_payment['start_date'] = pd.to_datetime(df_payment['start_date'])

# 2. '최초 계약 상품' 정보를 가져오기 위한 정렬 및 추출
df_sorted = df_payment.sort_values(['phone_number', 'start_date'])
first_product = df_sorted.groupby('phone_number')['product_name_clean'].first().reset_index()
first_product.columns = ['phone_number', 'first_product']

# 3. 유저별(phone_number) 집계 진행
df_user_cohort = df_payment.groupby('phone_number').agg(
    uid=('uid', 'first'),

    # [결제/재무 지표] - p_date 기준
    first_payment_date=('p_date', 'min'),
    total_revenue=('actual_price', lambda x: x[df_payment.loc[x.index, 'ph_status'] == 'DONE'].sum()),
    total_purchase_cnt=('ph_status', lambda x: (x == 'DONE').sum()),

    # [계약/이용 지표] - start_date 기준
    first_contract_date=('start_date', 'min'),
    total_contract_duration=('contract_duration', 'sum'),

    # [유저 성향/경험 지표]
    # 이미 df_payment에 유저별로 True/False가 계산되어 있으므로 'first'로 가져옵니다.
    is_count_pass_user=('is_count_pass_user', 'first'),
    has_refund_experience=('ph_status', lambda x: (x == 'CANCELED').any())
).reset_index()

# 4. 첫 구매 상품 정보 결합
df_user_cohort = df_user_cohort.merge(first_product, on='phone_number', how='left')

# 5. 코호트 월(Month) 컬럼 생성
df_user_cohort['cohort_month_contract'] = df_user_cohort['first_contract_date'].dt.to_period('M')
df_user_cohort['cohort_month_payment'] = df_user_cohort['first_payment_date'].dt.to_period('M')

# 6. 컬럼 순서 정렬 (UID, Phone, 코호트 기준 날짜들을 앞으로)
base_cols = ['uid', 'phone_number', 'cohort_month_contract', 'first_contract_date', 'first_payment_date']
other_cols = [c for c in df_user_cohort.columns if c not in base_cols]
df_user_cohort = df_user_cohort[base_cols + other_cols]

# 결과 확인
df_user_cohort.head()

## 체크인데이터

In [ ]:

# 2. SSH Tunnel 연결 및 DB 접속
with SSHTunnelForwarder(
    (ssh_host, 22),
    ssh_username=ssh_username,
    ssh_pkey=pem_path,
    remote_bind_address=(db_host, db_port),
    local_bind_address=('localhost', 5433)
) as tunnel:

    # 3. DB 연결 (로컬 포트를 통해)
    conn = psycopg2.connect(
        host='localhost',
        port=5433,
        database=db_name,
        user=db_user,
        password=db_password
    )

    # 4. 쿼리 실행
    df_check_in = pd.read_sql("""
                               SELECT
                                        TO_CHAR(ch.start_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS check_in_date,
                                        TO_CHAR(ch.start_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD HH24:MI:SS') AS check_in_Time,
                                        ch.check_in_history_uid,
                                        ci.contract_participant_uid,
                                        c.client_uid,
                                        c.contract_uid,
                                        ch.branch_uid,
                                        b.sub_type,
                                        b.display_name
                                FROM check_in ci
                                left join check_in_history ch
                                on ci.check_in_uid = ch.check_in_uid
                                left join branch b
                                on ch.branch_uid  = b.branch_uid
                                left join contract_participant cp
                                on cp.contract_participant_uid = ci.contract_participant_uid
                                left join contract c
                                on c.contract_uid = cp.contract_uid
                                LEFT JOIN price_policy pp
                                ON c.price_policy_uid = pp.price_policy_uid
                                where c.actual_price > 0

                    """, conn)
    conn.close()
df_check_in.head()

In [ ]:
import pandas as pd

# 1. start_date가 datetime 형식인지 다시 한번 확인 (안 되어있을 경우를 대비)
df_check_in['check_in_date'] = pd.to_datetime(df_check_in['check_in_date'])

# 2. 2025-06-11부터의 데이터만 남기기 (6월 10일 포함 이전 데이터 삭제)
# 기준일: 2025-06-11
df_check_in = df_check_in[df_check_in['check_in_date'] >= '2025-06-11'].reset_index(drop=True)
df_check_in = df_check_in[df_check_in['check_in_date'] <= '2026-04-30'].reset_index(drop=True)


#df 연결

In [ ]:
import pandas as pd

# ---------------------------------------------------------
# 1. 유저 마스터 집계 (df_payment 기준)
# ---------------------------------------------------------
df_user_cohort = df_payment.groupby('phone_number').agg(
    uid=('uid', 'first'),
    # [재무 지표]
    first_payment_date=('p_date', 'min'),
    total_revenue=('actual_price', lambda x: x[df_payment.loc[x.index, 'ph_status'] == 'DONE'].sum()),
    # ⭐ 총 환불액 추가
    total_refund_amount=('actual_price', lambda x: x[df_payment.loc[x.index, 'ph_status'] == 'CANCELED'].sum()),
    total_purchase_cnt=('ph_status', lambda x: (x == 'DONE').sum()),
    # [계약/이용 지표]
    first_contract_date=('start_date', 'min'),
    total_contract_duration=('contract_duration', 'sum'),
    # [유저 성향 및 환불 데이터]
    is_count_pass_user=('is_count_pass_user', 'first'),
    has_refund_experience=('ph_status', lambda x: (x == 'CANCELED').any()),
    days_before_cancel=('contract_duration', lambda x: x[df_payment.loc[x.index, 'ph_status'] == 'CANCELED'].mean())
).reset_index()

# 보너스: 순매출 계산
df_user_cohort['net_revenue'] = df_user_cohort['total_revenue'] - df_user_cohort['total_refund_amount']

# ---------------------------------------------------------
# 2. 지점 이용 행태 집계 (df_check_in 기준)
# ---------------------------------------------------------
df_check_in = df_check_in.rename(columns={'client_uid': 'uid'})
df_check_in['check_in_time'] = pd.to_datetime(df_check_in['check_in_time'])

# (1) 총 방문 및 지점별 방문수
total_visits = df_check_in.groupby('uid').size().reset_index(name='total_visits')
user_branch_agg = df_check_in.groupby(['uid', 'display_name']).size().reset_index(name='visit_count')

# (2) Main / First / Last Branch 추출
main_branch = user_branch_agg.sort_values(['uid', 'visit_count'], ascending=[True, False]).groupby('uid').first().reset_index()
main_branch = main_branch.rename(columns={'display_name': 'main_branch_name', 'visit_count': 'main_branch_visits'})

first_branch = df_check_in.sort_values(['uid', 'check_in_time']).groupby('uid').head(1)[['uid', 'display_name']].rename(columns={'display_name': 'first_branch_name'})
last_branch = df_check_in.sort_values(['uid', 'check_in_time']).groupby('uid').tail(1)[['uid', 'display_name']].rename(columns={'display_name': 'last_branch_name'})

# (3) 이용 지점 수
branch_variety = df_check_in.groupby('uid')['display_name'].nunique().reset_index(name='unique_branch_count')

# ---------------------------------------------------------
# 3. 유저 코호트 마스터와 모든 정보 결합
# ---------------------------------------------------------
df_final_raw = df_user_cohort.merge(total_visits, on='uid', how='left') \
                             .merge(first_branch, on='uid', how='left') \
                             .merge(main_branch, on='uid', how='left') \
                             .merge(last_branch, on='uid', how='left') \
                             .merge(branch_variety, on='uid', how='left')

# ---------------------------------------------------------
# 4. 분석 편의를 위한 결측치 처리
# ---------------------------------------------------------
fill_zero = ['total_visits', 'unique_branch_count', 'total_refund_amount', 'net_revenue', 'total_revenue']
df_final_raw[fill_zero] = df_final_raw[fill_zero].fillna(0)

fill_text = ['first_branch_name', 'main_branch_name', 'last_branch_name']
df_final_raw[fill_text] = df_final_raw[fill_text].fillna('방문기록없음')

# 결과 확인
df_final_raw.head()

In [ ]:
# 지점별 성적표 요약
branch_performance = df_final_raw.groupby('last_branch_name').agg(
    total_visitors=('uid', 'count'),
    refund_users=('has_refund_experience', 'sum')
).reset_index()

# 이탈 기여율(%) 계산
branch_performance['churn_rate_pct'] = (branch_performance['refund_users'] / branch_performance['total_visitors'] * 100).round(2)

# 방문객이 일정 수준(예: 5명) 이상인 지점 중 이탈율 높은 순 정렬
top_churn_branches = branch_performance[branch_performance['total_visitors'] >= 5].sort_values('churn_rate_pct', ascending=False)

print("⚠️ 이탈 유발 위험 지점 (Top 5)")
print(top_churn_branches.head(5))

In [ ]:
# 상위 20% 고거래액 유저 기준 산출
revenue_threshold = df_final_raw['total_revenue'].quantile(0.8)
vvip_users = df_final_raw[df_final_raw['total_revenue'] >= revenue_threshold]

vvip_branch_share = vvip_users.groupby('main_branch_name').agg(
    vvip_count=('uid', 'count'),
    totla_revenue=('total_revenue', 'sum')
).sort_values('vvip_count', ascending=False)

print("💰 매출 기여도가 가장 높은 핵심 지점")
print(vvip_branch_share.head(5))

In [ ]:
# 횟수권 유저와 무제한 유저의 지점 이용 행태 비교
product_branch_fit = df_final_raw.groupby(['is_count_pass_user', 'main_branch_name']).size().unstack(fill_value=0)

print("🔄 상품군별 선호 지점 차이 (상위 5개 지점)")
print(product_branch_fit.T.sort_values(by=True, ascending=False).head(5)) # 횟수권 유저 선호순

In [ ]:
vvip_branch_share = vvip_branch_share.reset_index()

In [ ]:
# 1. 일단 df_final_raw에 환불 기간 데이터가 있는지 강제로 다시 확인해서 넣어줍니다.
# (df_payment의 데이터를 uid 기준으로 가져와서 붙이는 방식입니다)
refund_period = df_payment[df_payment['ph_status'] == 'CANCELED'].groupby('uid')['contract_duration'].mean()
df_final_raw['days_before_cancel'] = df_final_raw['uid'].map(refund_period)

# 2. 이제 다시 분석 코드를 돌립니다.
refund_users = df_final_raw[df_final_raw['has_refund_experience'] == True]
fast_churn_branches = refund_users.groupby('last_branch_name').agg(
    refund_user_count=('uid', 'count'),
    avg_days_to_refund=('days_before_cancel', 'mean')
).reset_index()

# 확인
fast_churn_branches.head()

In [ ]:
# [공통 필터] 최소 분석 의미가 있는 지점 (방문자 5명 이상) 기준 설정
min_user_threshold = 5

# 🚀 INSIGHT 1. 지점별 '온보딩' 파워 (첫 지점이 미래를 결정한다)
# 유저별 최초 방문 지점 매칭
df_first_visit = df_check_in.sort_values(['uid', 'check_in_time']).groupby('uid').first().reset_index()
df_first_visit = df_first_visit[['uid', 'display_name']].rename(columns={'display_name': 'first_branch_name'})

df_onboarding = df_final_raw.merge(df_first_visit, on='uid', how='left')
top_onboarding_branches = df_onboarding.groupby('first_branch_name').agg(
    total_users=('uid', 'count'),
    refund_rate=('has_refund_experience', 'mean'),
    avg_revenue=('total_revenue', 'mean')
).reset_index()

# 필터링 및 점수화 (환불 적고 매출 높은 곳)
top_onboarding_branches = top_onboarding_branches[top_onboarding_branches['total_users'] >= min_user_threshold]
top_onboarding_branches['onboarding_score'] = (top_onboarding_branches['avg_revenue'] * (1 - top_onboarding_branches['refund_rate'])).round(0)
top_onboarding_branches = top_onboarding_branches.sort_values('onboarding_score', ascending=False)


# 🚀 INSIGHT 2. '노마드 정착' 베스트 지점 (유랑민들이 선택한 최종 목적지)
# 2개 이상 지점을 써본 유저 중 환불 안 한 우수 유저 대상
nomad_settlers = df_final_raw[(df_final_raw['unique_branch_count'] >= 2) & (df_final_raw['has_refund_experience'] == False)]
nomad_conversion_branches = nomad_settlers.groupby('last_branch_name').agg(
    settle_count=('uid', 'count'),
    avg_ltv=('total_revenue', 'mean')
).reset_index()

nomad_conversion_branches = nomad_conversion_branches[nomad_conversion_branches['settle_count'] >= 2] # 정착은 소수여도 의미있음
nomad_conversion_branches = nomad_conversion_branches.sort_values('settle_count', ascending=False)


# 🚀 INSIGHT 3. '실망의 속도' (이 지점은 오자마자 나간다?)
# 환불 유저들이 마지막 지점 이용 후 환불까지 걸린 시간
refund_users = df_final_raw[df_final_raw['has_refund_experience'] == True]
fast_churn_branches = refund_users.groupby('last_branch_name').agg(
    refund_user_count=('uid', 'count'),
    avg_days_to_refund=('days_before_cancel', 'mean') # 환불 전까지 이용 기간
).reset_index()

# 10일 이내 환불이 일어나는 '위험 지점' 순위 (모수 3명 이상)
fast_churn_branches = fast_churn_branches[fast_churn_branches['refund_user_count'] >= 3]
fast_churn_branches = fast_churn_branches.sort_values('avg_days_to_refund', ascending=True)

In [ ]:
fast_churn_branches.head()

In [ ]:
# 1. '방문기록없음' 유저 UID 추출
no_record_uids = df_final_raw[df_final_raw['main_branch_name'] == '방문기록없음']['uid'].unique()

# 2. 이 UID들이 df_check_in(원본)에 단 하나라도 있는지 확인
exists_in_checkin = df_check_in[df_check_in['uid'].isin(no_record_uids)]

# 3. 결과 요약 출력
print(f"📊 [검증 결과]")
print(f"- '방문기록없음' 유저 수: {len(no_record_uids)}명")
print(f"- 그 중 실제 원본(df_check_in)에 기록이 발견된 유저: {exists_in_checkin['uid'].nunique()}명")

# 4. 만약 실제 기록이 있는데 '방문기록없음'이 떴다면, 그 유저들 리스트 추출
if exists_in_checkin['uid'].nunique() > 0:
    print("\n⚠️ 주의: 기록이 있는데 누락된 유저가 있습니다. (로직 재점검 필요)")
    display(exists_in_checkin.head())
else:
    print("\n✅ 확인 완료: '방문기록없음' 유저들은 실제로 체크인 로그가 단 하나도 없습니다.")

In [ ]:
# 1. '방문기록없음' 유저의 UID 리스트 추출 (534명)
no_record_uids = df_final_raw[df_final_raw['main_branch_name'] == '방문기록없음']['uid'].unique()

# 2. 원본 df_payment 테이블에서 해당 유저들의 모든 결제 기록 필터링
df_방문기록없음 = df_payment[df_payment['uid'].isin(no_record_uids)].copy()

# 3. 제대로 담겼는지 확인 (유저 수와 데이터 행 수 출력)
print(f"✅ 추출된 유령 유저 수: {len(no_record_uids)}명")
print(f"✅ df_방문기록없음 데이터 행 수: {len(df_방문기록없음)}건")

# 4. (참고) 이들이 주로 어떤 상품을 샀는지 상위 5개 확인
print("\n📊 방문 기록 없는 유저들이 구매한 주요 상품:")
print(df_방문기록없음['product_name_clean'].value_counts().head(5))

# 결과 확인
df_방문기록없음.head()